# Đánh giá thiết kế đồ họa bằng Machine Learning
**Phiên bản nâng cấp** — Hybrid feature extraction: CLS token + Visual patches + Geometric metrics

# Data source: https://huggingface.co/datasets/creative-graphic-design/GraphicDesignEvaluation

In [1]:
import numpy as np
import torch
import io
from PIL import Image
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import pearsonr, spearmanr
import pytesseract
from transformers import LayoutLMv3Processor, LayoutLMv3Model
import joblib
import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Sử dụng thiết bị: {DEVICE}")


✅ Sử dụng thiết bị: cpu


## 1. Load dữ liệu — cả 3 chiều đánh giá

In [ ]:
from datasets import load_dataset

BASE = "hf://datasets/creative-graphic-design/GraphicDesignEvaluation"

print("⏳ Đang load 3 datasets (alignment / overlap / whitespace)...")

df_alignment  = pd.read_parquet(f"{BASE}/absolute-gpt-alignment/train-00000-of-00001.parquet")
df_overlap    = pd.read_parquet(f"{BASE}/absolute-gpt-overlap/train-00000-of-00001.parquet")
df_whitespace = pd.read_parquet(f"{BASE}/absolute-gpt-whitespace/train-00000-of-00001.parquet")

# Đặt tên label rõ ràng
df_alignment['label']  = df_alignment['avg'].astype(float)
df_overlap['label']    = df_overlap['avg'].astype(float)
df_whitespace['label'] = df_whitespace['avg'].astype(float)

print(f"   alignment  : {len(df_alignment):>4} mẫu  | label range: {df_alignment['label'].min():.1f} – {df_alignment['label'].max():.1f}")
print(f"   overlap    : {len(df_overlap):>4} mẫu  | label range: {df_overlap['label'].min():.1f} – {df_overlap['label'].max():.1f}")
print(f"   whitespace : {len(df_whitespace):>4} mẫu  | label range: {df_whitespace['label'].min():.1f} – {df_whitespace['label'].max():.1f}")
print("✅ Load xong!")


⏳ Đang load 3 datasets (alignment / overlap / whitespace)...


## 2. Khởi tạo LayoutLMv3

In [ ]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

print("⏳ Đang khởi tạo LayoutLMv3...")
layoutlm_processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=True
)
layoutlm_model = LayoutLMv3Model.from_pretrained("microsoft/layoutlmv3-base").to(DEVICE)
layoutlm_model.eval()
print("✅ LayoutLMv3 sẵn sàng!")


## 3. Hàm tính đặc trưng hình học (Geometric Features)

Đây là phần **cốt lõi bổ sung** — trích xuất trực tiếp từ toạ độ bounding box của các OCR token mà LayoutLMv3 đã phát hiện. Mỗi đặc trưng có tên và ý nghĩa rõ ràng.

In [ ]:
def compute_geometric_features(boxes_raw: np.ndarray) -> np.ndarray:
    """
    Tính 12 đặc trưng hình học từ bounding boxes của OCR tokens.
    boxes_raw: [N, 4] dạng [x0, y0, x1, y1] đã được normalize về [0, 1000]
               (LayoutLMv3 dùng scale 0-1000)

    Trả về vector 12 chiều với tên rõ ràng:
    ── Alignment (căn chỉnh) ──────────────────────────────
    [0] left_edge_std      : Độ lệch chuẩn cạnh trái  → nhỏ = căn đều
    [1] right_edge_std     : Độ lệch chuẩn cạnh phải
    [2] center_x_std       : Độ lệch chuẩn tâm ngang  → nhỏ = canh giữa đều
    [3] center_y_regularity: Khoảng cách dọc đều nhau → lớn = đều hàng
    ── Overlap (chồng lấp) ───────────────────────────────
    [4] overlap_ratio      : Tỉ lệ pixel chồng lấp / tổng canvas
    [5] max_iou            : IoU lớn nhất giữa 2 phần tử
    [6] overlap_pair_ratio : % cặp phần tử có chồng lấp
    ── Whitespace (không gian trắng) ─────────────────────
    [7]  whitespace_ratio  : Tỉ lệ diện tích trống / canvas
    [8]  margin_left       : Lề trái nhỏ nhất (normalize)
    [9]  margin_right      : Lề phải nhỏ nhất
    [10] margin_top        : Lề trên nhỏ nhất
    [11] density_variance  : Variance mật độ theo lưới 4x4
    """
    CANVAS = 1000.0  # LayoutLMv3 normalize bbox về 0-1000

    # Lọc bỏ padding boxes [0,0,0,0]
    valid = boxes_raw[(boxes_raw[:, 2] > boxes_raw[:, 0]) &
                      (boxes_raw[:, 3] > boxes_raw[:, 1])]

    if len(valid) < 2:
        return np.zeros(12, dtype=np.float32)

    x0, y0, x1, y1 = valid[:,0], valid[:,1], valid[:,2], valid[:,3]
    cx = (x0 + x1) / 2.0
    cy = (y0 + y1) / 2.0
    w  = (x1 - x0).clip(min=0)
    h  = (y1 - y0).clip(min=0)
    area = w * h

    # ── Alignment features ─────────────────────────────────────────
    rng_x0 = x0.max() - x0.min() + 1e-6
    rng_x1 = x1.max() - x1.min() + 1e-6
    rng_cx = cx.max() - cx.min() + 1e-6

    left_edge_std   = float(np.std(x0) / rng_x0)
    right_edge_std  = float(np.std(x1) / rng_x1)
    center_x_std    = float(np.std(cx) / rng_cx)

    # Regularity: khoảng cách dọc giữa các hàng có đều không
    sorted_cy = np.sort(np.unique(cy.round(-1)))  # làm tròn để nhóm cùng hàng
    if len(sorted_cy) >= 2:
        gaps = np.diff(sorted_cy)
        center_y_regularity = float(1.0 - np.std(gaps) / (np.mean(gaps) + 1e-6))
    else:
        center_y_regularity = 1.0

    # ── Overlap features ──────────────────────────────────────────
    n = len(valid)
    overlap_pixels = 0.0
    iou_list = []
    overlap_pairs = 0

    # Giới hạn N để tránh O(n²) quá lâu
    sample = valid[:min(n, 60)]
    ns = len(sample)
    sx0,sy0,sx1,sy1 = sample[:,0],sample[:,1],sample[:,2],sample[:,3]

    for i in range(ns):
        for j in range(i+1, ns):
            ix0 = max(sx0[i], sx0[j]);  iy0 = max(sy0[i], sy0[j])
            ix1 = min(sx1[i], sx1[j]);  iy1 = min(sy1[i], sy1[j])
            inter = max(0, ix1-ix0) * max(0, iy1-iy0)
            if inter > 0:
                a_i = (sx1[i]-sx0[i]) * (sy1[i]-sy0[i])
                a_j = (sx1[j]-sx0[j]) * (sy1[j]-sy0[j])
                union = a_i + a_j - inter
                iou_list.append(inter / (union + 1e-6))
                overlap_pixels += inter
                overlap_pairs += 1

    canvas_area = CANVAS * CANVAS
    overlap_ratio      = float(overlap_pixels / canvas_area)
    max_iou            = float(max(iou_list)) if iou_list else 0.0
    total_pairs        = ns * (ns - 1) / 2
    overlap_pair_ratio = float(overlap_pairs / (total_pairs + 1e-6))

    # ── Whitespace features ────────────────────────────────────────
    total_element_area = float(np.sum(area))
    whitespace_ratio   = float(1.0 - total_element_area / canvas_area)

    margin_left  = float(x0.min() / CANVAS)
    margin_right = float((CANVAS - x1.max()) / CANVAS)
    margin_top   = float(y0.min() / CANVAS)

    # Density grid 4×4
    grid_n = 4
    density = np.zeros((grid_n, grid_n))
    for k in range(len(valid)):
        col = min(int(cx[k] / CANVAS * grid_n), grid_n - 1)
        row = min(int(cy[k] / CANVAS * grid_n), grid_n - 1)
        density[row, col] += area[k] / canvas_area
    density_variance = float(np.var(density))

    feat = np.array([
        left_edge_std,       # [0]  alignment
        right_edge_std,      # [1]  alignment
        center_x_std,        # [2]  alignment
        center_y_regularity, # [3]  alignment
        overlap_ratio,       # [4]  overlap
        max_iou,             # [5]  overlap
        overlap_pair_ratio,  # [6]  overlap
        whitespace_ratio,    # [7]  whitespace
        margin_left,         # [8]  whitespace
        margin_right,        # [9]  whitespace
        margin_top,          # [10] whitespace
        density_variance,    # [11] whitespace
    ], dtype=np.float32)

    return feat


# Tên các đặc trưng để giải thích kết quả sau này
GEO_FEATURE_NAMES = [
    "geo_left_edge_std",
    "geo_right_edge_std",
    "geo_center_x_std",
    "geo_center_y_regularity",
    "geo_overlap_ratio",
    "geo_max_iou",
    "geo_overlap_pair_ratio",
    "geo_whitespace_ratio",
    "geo_margin_left",
    "geo_margin_right",
    "geo_margin_top",
    "geo_density_variance",
]

print(f"✅ Đã định nghĩa {len(GEO_FEATURE_NAMES)} geometric features với tên rõ ràng.")
print("   Nhóm Alignment :", GEO_FEATURE_NAMES[:4])
print("   Nhóm Overlap   :", GEO_FEATURE_NAMES[4:7])
print("   Nhóm Whitespace:", GEO_FEATURE_NAMES[7:])


## 4. Hàm trích xuất đặc trưng lai (Hybrid Feature Extraction)

**Ba phần song song:**
- **CLS token** (768d): ngữ nghĩa tổng thể của layout
- **Visual patch mean/std** (768+768d): phân bố không gian ảnh — LayoutLMv3 có 196 patch tokens ở cuối sequence
- **Geometric metrics** (12d): đặc trưng hình học tính trực tiếp từ OCR bboxes

In [ ]:
def extract_hybrid_features(df, batch_size=8, label_col='label'):
    """
    Trích xuất 3-part hybrid feature vector từ mỗi ảnh.

    Cấu trúc output vector (1548 chiều):
      [0:768]      CLS token      — ngữ nghĩa tổng thể (L2 normalized)
      [768:1536]   Visual patches — mean + std của 196 patch tokens (L2 norm)
      [1536:1548]  Geometric      — 12 chỉ số alignment/overlap/whitespace

    Lý do dùng visual patches thay vì chỉ CLS:
      LayoutLMv3 encode 14×14=196 image patches ở cuối sequence.
      Mean của chúng nắm bắt "phân bố không gian trung bình",
      Std của chúng nắm bắt "mức độ đa dạng bố cục" —
      cả hai đều liên quan trực tiếp đến whitespace và balance.
    """
    all_feat   = []
    all_labels = []
    n = len(df)

    print(f"\n🔄 Trích xuất hybrid features cho {n} mẫu...")

    for start in range(0, n, batch_size):
        end   = min(start + batch_size, n)
        batch = df.iloc[start:end]

        images = []
        for img in batch["image"]:
            if isinstance(img, dict):
                if img.get('bytes'):
                    img = Image.open(io.BytesIO(img['bytes']))
                elif img.get('path'):
                    img = Image.open(img['path'])
            elif type(img).__name__ == "ndarray":
                img = Image.fromarray(img)
            if img.mode != "RGB":
                img = img.convert("RGB")
            images.append(img)

        inputs = layoutlm_processor(
            images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(DEVICE)

        with torch.no_grad():
            outputs = layoutlm_model(**inputs)

        hidden = outputs.last_hidden_state  # [B, seq_len, 768]
        B, seq_len, D = hidden.shape

        # LayoutLMv3: text tokens chiếm seq_len-196 đầu, 196 patch tokens ở cuối
        N_PATCHES = 196
        text_end  = seq_len - N_PATCHES  # vị trí bắt đầu patch tokens

        for i in range(len(images)):
            # ── Phần 1: CLS token [0] ──────────────────────────────
            cls_vec = hidden[i, 0, :].cpu().float().numpy()
            cls_vec = cls_vec / (np.linalg.norm(cls_vec) + 1e-8)

            # ── Phần 2: Visual patch tokens ──────────────────────
            if text_end < seq_len:
                patch_tokens = hidden[i, text_end:, :].cpu().float().numpy()
            else:
                # Fallback: dùng tất cả tokens ngoài CLS
                patch_tokens = hidden[i, 1:, :].cpu().float().numpy()

            vis_mean = patch_tokens.mean(axis=0)
            vis_std  = patch_tokens.std(axis=0)
            vis_mean = vis_mean / (np.linalg.norm(vis_mean) + 1e-8)
            vis_std  = vis_std  / (np.linalg.norm(vis_std)  + 1e-8)
            visual_vec = np.concatenate([vis_mean, vis_std])  # 1536d

            # ── Phần 3: Geometric features từ OCR bboxes ─────────
            # inputs['bbox'] shape: [B, seq_len, 4] — toạ độ x0,y0,x1,y1 (0-1000)
            boxes = inputs['bbox'][i].cpu().numpy()           # [seq_len, 4]
            mask  = inputs['attention_mask'][i].cpu().numpy().astype(bool)
            valid_boxes = boxes[mask]                          # lọc padding
            geo_vec = compute_geometric_features(valid_boxes)  # 12d

            # ── Ghép lại ──────────────────────────────────────────
            feat = np.concatenate([cls_vec, visual_vec, geo_vec])
            all_feat.append(feat)

        labels = batch[label_col].values
        all_labels.extend(labels)

        print(f"   [{end:>4}/{n}] {end/n*100:5.1f}%  "
              f"feature_dim={all_feat[-1].shape[0]} ✓")

    X = np.vstack(all_feat)
    y = np.array(all_labels, dtype=np.float32)

    print(f"\n✅ Xong! X.shape={X.shape}, y.shape={y.shape}")
    print(f"   CLS:     dims [0:768]")
    print(f"   Visual:  dims [768:1536]")
    print(f"   Geo:     dims [1536:{X.shape[1]}]  ← có thể giải thích trực tiếp")
    return X, y


## 5. Trích xuất đặc trưng

Chạy trên dataset `alignment` (dataset chính). Bạn có thể thay bằng `df_overlap` hoặc `df_whitespace` để train model cho từng chiều.

In [ ]:
# Trích xuất cho alignment dataset
X, y = extract_hybrid_features(df_alignment, batch_size=16)

# Kiểm tra phân phối geometric features (12 chiều cuối)
geo_df = pd.DataFrame(X[:, -12:], columns=GEO_FEATURE_NAMES)
print("\n📊 Phân phối Geometric Features:")
print(geo_df.describe().round(4))
print("\n💡 Nếu tất cả các cột có std ≈ 0, OCR không phát hiện được chữ trong ảnh.")


## 6. Chia tập dữ liệu và chuẩn hóa

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

pca = PCA(n_components=100, random_state=42)
X_train_pca   = pca.fit_transform(X_train_sc[:, :-12])  # ✅ -12 thay vì :1536
X_test_pca    = pca.transform(X_test_sc[:, :-12])

X_train_final = np.hstack([X_train_pca, X_train_sc[:, -12:]])  # 100+12 = 112
X_test_final  = np.hstack([X_test_pca,  X_test_sc[:,  -12:]])

print(f"✅ Shape: {X_train_final.shape[1]} dims")  # phải in ra 112

## 7. Hàm đánh giá mô hình (có thêm R² và Correlation)

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    print(f"{'='*55}")
    print(f"🤖 MÔ HÌNH: {model_name}")
    print(f"{'='*55}")

    model.fit(X_tr, y_tr)

    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)

    # Clip về 0-10
    y_pred_tr = np.clip(y_pred_tr, 0, 10)
    y_pred_te = np.clip(y_pred_te, 0, 10)

    # Chỉ số hồi quy cơ bản
    metrics = {
        "Train MSE":  mean_squared_error(y_tr, y_pred_tr),
        "Test  MSE":  mean_squared_error(y_te, y_pred_te),
        "Train MAE":  mean_absolute_error(y_tr, y_pred_tr),
        "Test  MAE":  mean_absolute_error(y_te, y_pred_te),
        "Train R²":   r2_score(y_tr, y_pred_tr),
        "Test  R²":   r2_score(y_te, y_pred_te),
    }
    # Pearson & Spearman (quan trọng để so sánh với human/GPT)
    pearson_r,  pearson_p  = pearsonr(y_te, y_pred_te)
    spearman_r, spearman_p = spearmanr(y_te, y_pred_te)
    metrics["Pearson r"]  = pearson_r
    metrics["Spearman ρ"] = spearman_r

    for k, v in metrics.items():
        print(f"   {k:<14}: {v:>8.4f}")

    # Cảnh báo phân phối dự đoán
    pred_std = y_pred_te.std()
    true_std = y_te.std()
    print(f"\n   Std dự đoán  : {pred_std:.4f}  (thực tế: {true_std:.4f})")
    if pred_std < true_std * 0.4:
        print("   ⚠️  Dự đoán bị co cụm quá nhiều — model vẫn thiên về mean.")
    else:
        print("   ✅ Dự đoán có đủ độ phân tán.")

    # R² interpretation
    r2 = metrics["Test  R²"]
    if   r2 > 0.5:  print(f"   ✅ R²={r2:.3f} — Model giải thích được >50% variance")
    elif r2 > 0.2:  print(f"   🟡 R²={r2:.3f} — Model có học được nhưng còn yếu")
    else:           print(f"   🔴 R²={r2:.3f} — Model hầu như không học được pattern")

    # Ví dụ dự đoán
    print(f"\n   📋 10 mẫu test đầu tiên:")
    print(f"   {'Thực tế':>10} {'Dự đoán':>10} {'Sai lệch':>10}")
    print(f"   {'-'*32}")
    for a, p in zip(y_te[:10], y_pred_te[:10]):
        print(f"   {a:>10.4f} {p:>10.4f} {p-a:>+10.4f}")

    print()
    return model, metrics


## 8. Huấn luyện và so sánh các mô hình

In [ ]:
results = {}

# ── Mô hình 1: GradientBoosting (mạnh hơn RF với features chiều cao) ──
gb_model, gb_metrics = evaluate_model(
    GradientBoostingRegressor(
        n_estimators=300,
        max_depth=4,           # sâu hơn RF để học pattern phức tạp
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    ),
    X_train_final, X_test_final, y_train, y_test,
    "Gradient Boosting Regressor"
)
results["GradientBoosting"] = gb_metrics

# ── Mô hình 2: Random Forest (baseline, cải thiện max_depth) ──────────
rf_model, rf_metrics = evaluate_model(
    RandomForestRegressor(
        n_estimators=200,
        max_depth=8,           # tăng từ 2 → 8 để model học được nhiều hơn
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    ),
    X_train_final, X_test_final, y_train, y_test,
    "Random Forest (max_depth=8)"
)
results["RandomForest"] = rf_metrics

# ── Mô hình 3: SVR ─────────────────────────────────────────────────────
svr_model, svr_metrics = evaluate_model(
    SVR(kernel="rbf", C=5, epsilon=0.05, gamma="scale"),
    X_train_final, X_test_final, y_train, y_test,
    "SVR (RBF, C=5)"
)
results["SVR"] = svr_metrics

# Tổng kết
print("\n" + "="*55)
print("📊 SO SÁNH CÁC MÔ HÌNH (Test set)")
print("="*55)
print(f"{'Model':<25} {'R²':>8} {'MAE':>8} {'Pearson':>10}")
print("-"*55)
for name, m in results.items():
    print(f"{name:<25} {m['Test  R²']:>8.4f} {m['Test  MAE']:>8.4f} {m['Pearson r']:>10.4f}")


## 9. Phân tích Feature Importance — Mô hình học được gì?

Đây là điểm khác biệt cốt lõi: ta có thể kiểm tra xem geometric features nào đóng góp nhiều nhất.

In [ ]:
# Tên đầy đủ của 112 features
pca_names = [f"pca_{i:03d}" for i in range(100)]
feature_names = pca_names + GEO_FEATURE_NAMES   # 100 + 12 = 112

# Feature importance từ GradientBoosting
gb_importance = pd.Series(
    gb_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

print("🔍 TOP 15 features quan trọng nhất (GradientBoosting):")
print("-" * 45)
for feat, imp in gb_importance.head(15).items():
    bar = "█" * int(imp * 500)
    tag = " ← GEO!" if feat.startswith("geo_") else ""
    print(f"  {feat:<28} {imp:.4f}  {bar}{tag}")

print("\n📌 Importance của từng Geometric Feature:")
print("-" * 45)
geo_imp = gb_importance[gb_importance.index.str.startswith("geo_")]
for feat, imp in geo_imp.items():
    rank = list(gb_importance.index).index(feat) + 1
    print(f"  Rank #{rank:<3} {feat:<28} {imp:.4f}")

print("\n💡 Nếu geo features chiếm top rank → chúng là signal quan trọng.")
print("   Nếu chỉ có pca_xxx → model vẫn đang dựa vào deep features chung chung.")


## 10. Phân tích tương quan Geometric Features với nhãn

In [ ]:
# Lấy 12 geo features từ tập train
geo_train = X_train_sc[:, -12:]    # luôn đúng 12 geo features
geo_test  = X_test_sc[:,  -12:]

print("📊 Tương quan Pearson giữa Geometric Features và nhãn alignment:")
print(f"{'Feature':<28} {'Pearson r':>10} {'|r|':>8} {'Ý nghĩa'}")
print("-" * 75)

correlations = {}
for i, name in enumerate(GEO_FEATURE_NAMES):
    r, p = pearsonr(geo_train[:, i], y_train)
    correlations[name] = r
    sig = "✅ Tương quan tốt" if abs(r) > 0.2 else ("🟡 Yếu" if abs(r) > 0.1 else "⬜ Rất yếu")
    print(f"  {name:<28} {r:>10.4f} {abs(r):>8.4f}  {sig}")

best = max(correlations, key=lambda k: abs(correlations[k]))
print(f"\n🏆 Geometric feature tương quan mạnh nhất: {best} (r={correlations[best]:.4f})")


## 11. Lưu model và hàm chấm điểm ảnh mới

In [ ]:
# Lưu artifacts
joblib.dump(gb_model,  "gb_model.pkl")
joblib.dump(rf_model,  "rf_model.pkl")
joblib.dump(svr_model, "svr_model.pkl")
joblib.dump(scaler,    "scaler.pkl")
joblib.dump(pca,       "pca.pkl")
print("✅ Đã lưu: gb_model | rf_model | svr_model | scaler | pca")


In [ ]:
def score_design(image_source):
    """
    Chấm điểm một ảnh thiết kế mới.
    Trả về dict gồm score và giải thích từng geometric feature.
    """
    # Load ảnh
    if isinstance(image_source, str):
        if image_source.startswith("http"):
            import urllib.request
            with urllib.request.urlopen(image_source) as r:
                img = Image.open(io.BytesIO(r.read()))
        else:
            img = Image.open(image_source)
    elif isinstance(image_source, Image.Image):
        img = image_source
    else:
        raise TypeError("Cần đường dẫn, URL hoặc PIL Image")

    if img.mode != "RGB":
        img = img.convert("RGB")
    w, h = img.size
    if w < 224 or h < 224:
        scale = max(224/w, 224/h)
        img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)

    print(f"🖼️  Kích thước: {img.size} | Mode: {img.mode}")
    print("⏳ Đang trích xuất hybrid features...")

    inputs = layoutlm_processor(
        img, return_tensors="pt",
        padding=True, truncation=True, max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        outputs = layoutlm_model(**inputs)

    hidden = outputs.last_hidden_state
    seq_len = hidden.shape[1]
    N_PATCHES = 196
    text_end = seq_len - N_PATCHES

    cls_vec = hidden[0, 0, :].cpu().float().numpy()
    cls_vec = cls_vec / (np.linalg.norm(cls_vec) + 1e-8)

    patch_tokens = hidden[0, text_end:, :].cpu().float().numpy()
    vis_mean = patch_tokens.mean(axis=0); vis_mean /= (np.linalg.norm(vis_mean)+1e-8)
    vis_std  = patch_tokens.std(axis=0);  vis_std  /= (np.linalg.norm(vis_std)+1e-8)

    boxes = inputs['bbox'][0].cpu().numpy()
    mask  = inputs['attention_mask'][0].cpu().numpy().astype(bool)
    geo_vec = compute_geometric_features(boxes[mask])

    feat_raw = np.concatenate([cls_vec, vis_mean, vis_std, geo_vec]).reshape(1, -1)
    feat_sc  = scaler.transform(feat_raw)

    feat_pca = pca.transform(feat_sc[:, :1536])
    feat_final = np.hstack([feat_pca, feat_sc[:, 1536:]])

    score_gb  = float(np.clip(gb_model.predict(feat_final)[0],  0, 10))
    score_rf  = float(np.clip(rf_model.predict(feat_final)[0],  0, 10))
    score_svr = float(np.clip(svr_model.predict(feat_final)[0], 0, 10))
    score_avg = round((score_gb + score_rf + score_svr) / 3, 2)

    verdict_map = [(8,"🟢 Xuất sắc"), (6.5,"🟡 Tốt"), (5,"🟠 Trung bình"), (0,"🔴 Cần cải thiện")]
    verdict = next(v for t,v in verdict_map if score_avg >= t)

    print("\n" + "="*45)
    print("📐 KẾT QUẢ CHẤM ĐIỂM")
    print("="*45)
    print(f"   Gradient Boosting : {score_gb:.2f} / 10")
    print(f"   Random Forest     : {score_rf:.2f} / 10")
    print(f"   SVR               : {score_svr:.2f} / 10")
    print(f"   ── Trung bình     : {score_avg:.2f} / 10  ◀")
    print(f"   Đánh giá          : {verdict}")

    print("\n🔍 Giải thích Geometric Features (có thể giải thích):")
    geo_labels = {
        "geo_left_edge_std":       ("Alignment",  "Cạnh trái lệch nhiều = bố cục kém căn chỉnh"),
        "geo_center_x_std":        ("Alignment",  "Tâm ngang lệch = thiếu trục căn chỉnh"),
        "geo_center_y_regularity": ("Alignment",  "Khoảng cách hàng đều = nhịp điệu tốt"),
        "geo_overlap_ratio":       ("Overlap",    "Tỉ lệ chồng lấp pixel"),
        "geo_max_iou":             ("Overlap",    "Cặp phần tử chồng lấp nặng nhất"),
        "geo_whitespace_ratio":    ("Whitespace", "Tỉ lệ không gian trắng"),
        "geo_margin_left":         ("Whitespace", "Lề trái"),
        "geo_density_variance":    ("Whitespace", "Phân bố mật độ đều hay không"),
    }
    for i, name in enumerate(GEO_FEATURE_NAMES):
        val = geo_vec[i]
        if name in geo_labels:
            group, desc = geo_labels[name]
            print(f"   [{group:<10}] {name:<28}: {val:.4f}  ({desc})")

    return {"score_gb": score_gb, "score_rf": score_rf, "score_svr": score_svr,
            "score_avg": score_avg, "verdict": verdict, "geo_features": dict(zip(GEO_FEATURE_NAMES, geo_vec))}


In [ ]:
# ── Sử dụng ──
result = score_design("a64386de4cebdfee327a4a544a3dd05d.jpg")